# Portfolio assignment week 2

Last week, you have been experimenting with the interaction of hyperparameters. You made visualisations to show how they impacted each other. This week, you will extend the number of hyperparameters and architectures you can experiment with.

1. Study more layers to add
Study the pytorch documentation for:

Dropout https://pytorch.org/docs/stable/generated/torch.nn.Dropout.html
normalization layers https://pytorch.org/docs/stable/nn.html#normalization-layers

2. Add dropout and normalization layers to your model
Experiment with adding dropout and normalization layers to your model. Some rough guidelines where to add them relative to Linear or Conv2d layers:

Dropout: after Linear or Conv2d layers. Often added after the last Linear layer before the output layer, but could occur more often.
Normalization layers: right after (blocks of) Linear or Conv2d layers, but before activation functions.

3. Use logging
set up logging with MLflow, and make sure the hyperparameters you are using are logged.
get comfortable with using MLflow to visualize your results, it has a pretty powerful dashboard.

4. Adding convolutional and pooling layers
This lesson, we have added some new types of layers: convolutional and pooling layers. Experiment with adding these new layers.

Also, have a look at the ModuleList: https://pytorch.org/docs/stable/generated/torch.nn.ModuleList.html#modulelist This makes it much easier to use the number of layers as a hyperparameter. You can create a list of layers from a config, and then use that list to create your model. Instead of just adding a single layer by hand, you could also add a block of layers as a unit (eg a Conv2d layer, followed by a ReLU layer, followed by a BatchNorm2d layer, followed by a MaxPool2d layer) and repeat that in a loop, adding it to the ModuleList.

5. Reflect
Doing a master means you don't just start engineering a pipeline, but you need to reflect. Why do you see the results you see? What does this mean, considering the theory? Write down lessons learned and reflections, based on experimental results. This is the science part of data science.

You follow this cycle:

make a hypothesis
design an experiment
run the experiment
analyze the results and draw conclusions
repeat
To keep track of this process, it is useful to keep a journal. While you could use anything to do so, a nice command line tool is jrnl. This gives you the advantage of staying in the terminal, just type down your ideas during the process, and you can always look back at what you have done. Try to first formulate a hypothesis, and then design an experiment to test it. This will help you to stay focused on the goal, and not get lost in the data.

Important: the report you write is NOT the same as your journal! The journal will help you to keep track of your process, and later write down a reflection on what you have done where you draw conclusion, reflecting back on the theory.

6. Make a short report
Make a short 1 a4 page report of your findings. pay attention to:

what was your hypothesis about interaction between hyperparameters?
what did you find?
visualise your results about the relationship between hyperparameters.

In [1]:
from pathlib import Path
import warnings
warnings.simplefilter("ignore", UserWarning)

import torch
import torch.nn as nn
import torch.optim as optim
from loguru import logger

from mads_datasets import DatasetFactoryProvider, DatasetType
from mltrainer.preprocessors import BasePreprocessor
from mltrainer import metrics, Trainer, TrainerSettings, ReportTypes

Step 1: Data

In [2]:
fashionfactory = DatasetFactoryProvider.create_factory(DatasetType.FASHION)
batchsize = 64
preprocessor = BasePreprocessor()
streamers = fashionfactory.create_datastreamer(batchsize=batchsize, preprocessor=preprocessor)
train = streamers["train"]
valid = streamers["valid"]
trainstreamer = train.stream()
validstreamer = valid.stream()

x, y = next(iter(trainstreamer))
print(x.shape, y.shape)  # (batch, channel, width, height) en labels

2026-02-24 16:23:57.397 | INFO     | mads_datasets.base:download_data:121 - Folder already exists at /home/jgerrits/.cache/mads_datasets/fashionmnist
2026-02-24 16:23:57.399 | INFO     | mads_datasets.base:download_data:124 - File already exists at /home/jgerrits/.cache/mads_datasets/fashionmnist/fashionmnist.pt


torch.Size([64, 1, 28, 28]) torch.Size([64])


Step 2: Device

In [3]:
if torch.backends.mps.is_available() and torch.backends.mps.is_built():
    device = torch.device("mps")
    print("Using MPS")
elif torch.cuda.is_available():
    device = torch.device("cuda:0")
    print("Using CUDA")
else:
    device = torch.device("cpu")
    print("Using CPU")

print(f"Using {device} device")

Using CPU
Using cpu device


Step 3: Model architecture

In [4]:
class CNN(nn.Module):
    def __init__(self, filters, units1, units2, dropout_p=0.0, input_size=(32, 1, 28, 28)):
        super().__init__()
        self.in_channels = input_size[1]
        self.input_size = input_size
        self.filters = filters
        self.units1 = units1
        self.units2 = units2
        self.dropout_p = dropout_p  #TOML

        self.convolutions = nn.Sequential(
            nn.Conv2d(self.in_channels, filters, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(filters, filters, kernel_size=3, stride=1, padding=0),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(filters, filters, kernel_size=3, stride=1, padding=0),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        )

        activation_map_size = self._conv_test(input_size)
        logger.info(f"Aggregating activationmap with size {activation_map_size}")
        self.agg = nn.AvgPool2d(activation_map_size)

        # Dropout in dense stack 
        self.dense = nn.Sequential(
            nn.Flatten(),
            nn.Linear(filters, units1),
            nn.ReLU(),
            nn.Dropout(p=dropout_p),          
            nn.Linear(units1, units2),
            nn.ReLU(),
            nn.Dropout(p=dropout_p),         
            nn.Linear(units2, 10)
        )

    def _conv_test(self, input_size=(32, 1, 28, 28)):
        x = torch.ones(input_size)
        x = self.convolutions(x)
        return x.shape[-2:]

    def forward(self, x):
        x = self.convolutions(x)
        x = self.agg(x)
        logits = self.dense(x)
        return logits


Step 4: Loss metrics

In [8]:
accuracy = metrics.Accuracy()
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = optim.Adam

# MLflow + TOML logging
settings = TrainerSettings(
    epochs=10,  
    metrics=[accuracy],
    logdir="modellogs_dropout_h1",
    train_steps=200,   
    valid_steps=200,
    reporttypes=[ReportTypes.MLFLOW, ReportTypes.TOML],
)

Step 5: MLFlow setup

In [9]:
import mlflow
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("fashion_cnn_dropout_h1_larger_modelsize")

2026/02/24 16:48:09 INFO mlflow.tracking.fluent: Experiment with name 'fashion_cnn_dropout_h1_larger_modelsize' does not exist. Creating a new experiment.


<Experiment: artifact_location='/home/jgerrits/portfolio-S3/2-hypertuning-mlflow/mlruns/2', creation_time=1771948089568, experiment_id='2', last_update_time=1771948089568, lifecycle_stage='active', name='fashion_cnn_dropout_h1_larger_modelsize', tags={}>

Step 6: Experiment loop

Notes: Met hulp van chatgpt een for-loop gebruikt in plaats van de Hyperopt uit de code van de studie


In [10]:
dropout_ps = [0.0, 0.2, 0.5]

filters = 64
units1 = 256
units2 = 128

for p in dropout_ps:
    run_name = f"largemodel_dropout_p={p}_f={filters}_u1={units1}_u2={units2}"

    with mlflow.start_run(run_name=run_name):
        # log hyperparams
        mlflow.log_params({
            "dropout_p": p,
            "filters": filters,
            "units1": units1,
            "units2": units2,
            "epochs": settings.epochs,
            "batchsize": batchsize,
            "train_steps": settings.train_steps,
            "valid_steps": settings.valid_steps,
            "optimizer": "Adam",
            "loss": "CrossEntropyLoss",
        })

        model = CNN(filters=filters, units1=units1, units2=units2, dropout_p=p).to(device)

        trainer = Trainer(
            model=model,
            settings=settings,
            loss_fn=loss_fn,
            optimizer=optimizer,
            traindataloader=trainstreamer,
            validdataloader=validstreamer,
            scheduler=optim.lr_scheduler.ReduceLROnPlateau,
            device=device,
        )

        trainer.loop()


        for attr_name in ["train_loss", "valid_loss", "test_loss", "train_acc", "valid_acc", "test_acc"]:
            if hasattr(trainer, attr_name):
                value = getattr(trainer, attr_name)
                if isinstance(value, (int, float)):
                    mlflow.log_metric(attr_name, value)

        if hasattr(trainer, "train_acc") and hasattr(trainer, "valid_acc"):
            try:
                gap = float(trainer.train_acc) - float(trainer.valid_acc)
                mlflow.log_metric("generalization_gap_acc", gap)
            except Exception:
                pass

print("Done")

2026-02-24 16:48:26.365 | INFO     | __main__:__init__:26 - Aggregating activationmap with size torch.Size([2, 2])
2026-02-24 16:48:26.367 | INFO     | mltrainer.trainer:dir_add_timestamp:24 - Logging to modellogs_dropout_h1/20260224-164826
2026-02-24 16:48:26.368 | INFO     | mltrainer.trainer:__init__:68 - Found earlystop_kwargs in settings.Set to None if you dont want earlystopping.
100%|██████████| 200/200 [00:12<00:00, 15.49it/s]
2026-02-24 16:48:46.410 | INFO     | mltrainer.trainer:report:209 - Epoch 0 train 1.1788 test 0.8397 metric ['0.6769']
100%|██████████| 200/200 [00:12<00:00, 15.78it/s]
2026-02-24 16:49:07.185 | INFO     | mltrainer.trainer:report:209 - Epoch 1 train 0.7267 test 0.6824 metric ['0.7337']
100%|██████████| 200/200 [00:13<00:00, 14.41it/s]
2026-02-24 16:49:28.164 | INFO     | mltrainer.trainer:report:209 - Epoch 2 train 0.6277 test 0.6163 metric ['0.7612']
100%|██████████| 200/200 [00:12<00:00, 15.79it/s]
2026-02-24 16:49:47.942 | INFO     | mltrainer.trainer

Done
